# Agent 1 — SyllabusStore: PostgreSQL + Qdrant

## Purpose

This notebook creates a **single syllabus storage and lookup layer** for Agent 1.

```text
Module 1 / Module 3 / HITL / Frontend
                 |
                 v
           SyllabusStore
          /             \
         v               v
   PostgreSQL          Qdrant
 structured source   semantic index
```

### Storage rule

- **PostgreSQL (`public.syllabus_concepts`) is the authoritative source of truth.**
- **Qdrant is the semantic vector index derived from PostgreSQL.**
- The old static `cs_concept_catalog.py` is **not imported by SyllabusStore**.
- Existing topic-mapping, scoring, ambiguity, HITL and self-improving logic are **not moved into this store**.

The 92 catalogue concepts and their matching rules have already been migrated into PostgreSQL. This notebook now creates the reusable access layer that will eventually replace all direct catalogue lookups.


## What this notebook does

1. Locates the Agent 1 project root.
2. Creates `app/services/syllabus_store.py`.
3. Provides PostgreSQL structured lookups: `get_concept()`, `get_all_concepts()`, `get_concepts_by_reference()`, `get_concepts_by_paper()`, and `get_technical_terms()`.
4. Provides PostgreSQL storage through `upsert_concept()`.
5. Provides Qdrant functions for full sync, one-concept sync, semantic search, vector retrieval, and PostgreSQL/Qdrant consistency checks.
6. Runs smoke tests against the migrated 92-concept table.

### What it does **not** change

- primary/supporting classification
- lexical scoring
- evidence quality
- confidence calculation
- ambiguity decisions
- conflicting-context penalties
- parent suppression
- topic merging
- HITL decisions
- self-improving memory

Those behaviours remain where they already work. SyllabusStore only provides the data they need.


## Phase 1 — Locate the project and validate the environment

The notebook searches upward for the main `Agent_1` folder containing the `app/` package, so it can still work when opened from a `Notebooks/` folder.


In [ ]:
from pathlib import Path
import sys

def find_agent1_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "app").is_dir() and (candidate / "app" / "db").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Could not locate the Agent_1 project root. "
        "Open this notebook from inside the Agent_1 project."
    )

PROJECT_ROOT = find_agent1_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Agent 1 root:", PROJECT_ROOT)
print("app package:", PROJECT_ROOT / "app")
print("Syllabus table expected: public.syllabus_concepts")


## Phase 2 — Create the production `SyllabusStore` module

The next cell writes `Agent_1/app/services/syllabus_store.py`.

### Why keep the old object shape?

The new `SyllabusConcept` exposes familiar fields such as `concept_id`, `label`, `aliases`, `description`, `paper`, `parent_concept_id`, and `conflicting_context_terms`. This lets us change **where the concept comes from** without rewriting the already-working mapping logic.

### PostgreSQL is authoritative

Every structured concept lookup comes from `public.syllabus_concepts`.

### Qdrant is derived

Qdrant gets its embedding text and concept IDs from PostgreSQL, so the static catalogue is no longer needed to rebuild the semantic index.


In [ ]:
MODULE_SOURCE = 'from __future__ import annotations\n\nimport json\nimport os\nimport uuid\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom typing import Any, Iterable, Literal\n\nimport numpy as np\nfrom qdrant_client import QdrantClient, models\nfrom sqlalchemy import Engine, text\n\nfrom app.db.session import get_engine, load_environment\nfrom app.services.embedding_service import (\n    TOPIC_EMBEDDING_MODEL,\n    embed_texts,\n    get_embedding_dimension,\n)\n\n\nPaper = Literal["Paper 1", "Paper 2"]\nDEFAULT_QDRANT_URL = "http://localhost:6333"\nDEFAULT_COLLECTION_NAME = "aqa_gcse_computer_science_8525"\nDEFAULT_SPECIFICATION_CODE = "8525"\nDEFAULT_SPECIFICATION_VERSION = "AQA-8525-v1.2-2022-11-29"\n\n# Keep the namespace already used by the old qdrant_syllabus_store.py so\n# existing concept point IDs stay deterministic across the refactor.\nQDRANT_POINT_NAMESPACE = uuid.UUID(\n    "27a6c1e1-01bb-4fdc-88b8-e89d87c85425"\n)\n\n\n@dataclass(frozen=True)\nclass FlexiblePattern:\n    """One flexible lexical pattern stored in PostgreSQL as JSONB."""\n\n    label: str\n    regex: str\n    weight: float = 0.82\n\n\n@dataclass(frozen=True)\nclass SyllabusConcept:\n    """\n    Runtime representation of one row from public.syllabus_concepts.\n\n    Its public field names intentionally mirror the old CSConcept shape so\n    Agent 1 can be migrated with minimal downstream logic changes.\n    """\n\n    concept_id: str\n    official_reference: str\n    chapter_reference: str\n    chapter_title: str\n    official_title: str\n    label: str\n    domain: str\n    description: str\n    aliases: tuple[str, ...]\n    paper: Paper\n    source_pages: tuple[int, ...]\n    parent_concept_id: str | None = None\n    excluded_phrases: tuple[str, ...] = ()\n    ambiguous_aliases: tuple[str, ...] = ()\n    supporting_context_terms: tuple[str, ...] = ()\n    conflicting_context_terms: tuple[str, ...] = ()\n    minimum_context_hits: int = 1\n    match_patterns: tuple[FlexiblePattern, ...] = ()\n    embedding_text: str = ""\n    specification_code: str = DEFAULT_SPECIFICATION_CODE\n    specification_version: str = DEFAULT_SPECIFICATION_VERSION\n    is_active: bool = True\n\n\n@dataclass(frozen=True)\nclass SemanticConceptMatch:\n    """One nearest-neighbour result returned by Qdrant."""\n\n    concept_id: str\n    score: float\n    payload: dict[str, Any]\n\n\n@dataclass(frozen=True)\nclass SyllabusStoreConfig:\n    """Configuration for PostgreSQL-backed syllabus data and Qdrant search."""\n\n    specification_code: str = DEFAULT_SPECIFICATION_CODE\n    specification_version: str = DEFAULT_SPECIFICATION_VERSION\n    qdrant_url: str = DEFAULT_QDRANT_URL\n    qdrant_collection: str = DEFAULT_COLLECTION_NAME\n    qdrant_api_key: str | None = None\n    qdrant_timeout_seconds: float = 30.0\n    qdrant_top_k: int = 20\n    qdrant_score_threshold: float | None = None\n    embedding_model: str = TOPIC_EMBEDDING_MODEL\n\n    @classmethod\n    def from_environment(cls) -> "SyllabusStoreConfig":\n        load_environment()\n\n        threshold_text = os.getenv("QDRANT_SCORE_THRESHOLD", "").strip()\n        threshold = float(threshold_text) if threshold_text else None\n\n        return cls(\n            specification_code=os.getenv(\n                "AQA_SPECIFICATION_CODE", DEFAULT_SPECIFICATION_CODE\n            ).strip(),\n            specification_version=os.getenv(\n                "AQA_SPEC_VERSION", DEFAULT_SPECIFICATION_VERSION\n            ).strip(),\n            qdrant_url=os.getenv("QDRANT_URL", DEFAULT_QDRANT_URL).strip(),\n            qdrant_collection=os.getenv(\n                "QDRANT_COLLECTION", DEFAULT_COLLECTION_NAME\n            ).strip(),\n            qdrant_api_key=os.getenv("QDRANT_API_KEY", "").strip() or None,\n            qdrant_timeout_seconds=float(\n                os.getenv("QDRANT_TIMEOUT_SECONDS", "30")\n            ),\n            qdrant_top_k=int(os.getenv("QDRANT_TOP_K_PER_UNIT", "20")),\n            qdrant_score_threshold=threshold,\n            embedding_model=os.getenv(\n                "TOPIC_EMBEDDING_MODEL", TOPIC_EMBEDDING_MODEL\n            ).strip(),\n        )\n\n    def __post_init__(self) -> None:\n        if not self.specification_code:\n            raise ValueError("specification_code cannot be empty")\n        if not self.specification_version:\n            raise ValueError("specification_version cannot be empty")\n        if not self.qdrant_url:\n            raise ValueError("qdrant_url cannot be empty")\n        if not self.qdrant_collection:\n            raise ValueError("qdrant_collection cannot be empty")\n        if self.qdrant_timeout_seconds <= 0:\n            raise ValueError("qdrant_timeout_seconds must be positive")\n        if self.qdrant_top_k < 1:\n            raise ValueError("qdrant_top_k must be at least 1")\n        if (\n            self.qdrant_score_threshold is not None\n            and not -1.0 <= self.qdrant_score_threshold <= 1.0\n        ):\n            raise ValueError("qdrant_score_threshold must be between -1 and 1")\n\n\nclass SyllabusStore:\n    """\n    Single gateway for Agent 1 syllabus storage and lookup.\n\n    PostgreSQL is the authoritative structured store.\n    Qdrant is the semantic vector index rebuilt from PostgreSQL.\n\n    This class intentionally does NOT decide whether a topic is primary,\n    supporting, ambiguous, accepted or rejected. Existing Module 3 / HITL\n    logic continues to make those decisions using metadata returned here.\n    """\n\n    TABLE_NAME = "public.syllabus_concepts"\n\n    def __init__(\n        self,\n        config: SyllabusStoreConfig | None = None,\n        *,\n        engine: Engine | None = None,\n        qdrant_client: QdrantClient | None = None,\n    ) -> None:\n        self.config = config or SyllabusStoreConfig.from_environment()\n        self.engine = engine or get_engine()\n        self._qdrant_client = qdrant_client\n\n    # ------------------------------------------------------------------\n    # PostgreSQL deserialisation helpers\n    # ------------------------------------------------------------------\n\n    @staticmethod\n    def _json_value(value: Any, default: Any) -> Any:\n        if value is None:\n            return default\n        if isinstance(value, str):\n            stripped = value.strip()\n            if not stripped:\n                return default\n            return json.loads(stripped)\n        return value\n\n    @classmethod\n    def _row_to_concept(cls, row: Any) -> SyllabusConcept:\n        data = dict(row._mapping if hasattr(row, "_mapping") else row)\n\n        aliases = cls._json_value(data.get("aliases"), [])\n        source_pages = cls._json_value(data.get("source_pages"), [])\n        excluded = cls._json_value(data.get("excluded_phrases"), [])\n        ambiguous = cls._json_value(data.get("ambiguous_aliases"), [])\n        supporting = cls._json_value(data.get("supporting_context_terms"), [])\n        conflicting = cls._json_value(data.get("conflicting_context_terms"), [])\n        raw_patterns = cls._json_value(data.get("match_patterns"), [])\n\n        patterns = tuple(\n            FlexiblePattern(\n                label=str(item["label"]),\n                regex=str(item["regex"]),\n                weight=float(item.get("weight", 0.82)),\n            )\n            for item in raw_patterns\n        )\n\n        return SyllabusConcept(\n            concept_id=str(data["concept_id"]),\n            official_reference=str(data["official_reference"]),\n            chapter_reference=str(data["chapter_reference"]),\n            chapter_title=str(data["chapter_title"]),\n            official_title=str(data["official_title"]),\n            label=str(data["label"]),\n            domain=str(data["domain"]),\n            description=str(data["description"]),\n            aliases=tuple(str(v) for v in aliases),\n            paper=str(data["paper"]),\n            source_pages=tuple(int(v) for v in source_pages),\n            parent_concept_id=(\n                str(data["parent_concept_id"])\n                if data.get("parent_concept_id")\n                else None\n            ),\n            excluded_phrases=tuple(str(v) for v in excluded),\n            ambiguous_aliases=tuple(str(v) for v in ambiguous),\n            supporting_context_terms=tuple(str(v) for v in supporting),\n            conflicting_context_terms=tuple(str(v) for v in conflicting),\n            minimum_context_hits=int(data.get("minimum_context_hits") or 1),\n            match_patterns=patterns,\n            embedding_text=str(data.get("embedding_text") or ""),\n            specification_code=str(\n                data.get("specification_code") or DEFAULT_SPECIFICATION_CODE\n            ),\n            specification_version=str(\n                data.get("specification_version")\n                or DEFAULT_SPECIFICATION_VERSION\n            ),\n            is_active=bool(data.get("is_active", True)),\n        )\n\n    @staticmethod\n    def _base_select() -> str:\n        return """\n            SELECT\n                concept_id,\n                official_reference,\n                chapter_reference,\n                chapter_title,\n                official_title,\n                label,\n                domain,\n                description,\n                aliases,\n                paper,\n                source_pages,\n                parent_concept_id,\n                excluded_phrases,\n                ambiguous_aliases,\n                supporting_context_terms,\n                conflicting_context_terms,\n                minimum_context_hits,\n                match_patterns,\n                embedding_text,\n                specification_code,\n                specification_version,\n                is_active\n            FROM public.syllabus_concepts\n        """\n\n    def _spec_params(self) -> dict[str, str]:\n        return {\n            "specification_code": self.config.specification_code,\n            "specification_version": self.config.specification_version,\n        }\n\n    # ------------------------------------------------------------------\n    # PostgreSQL reads: authoritative structured lookup\n    # ------------------------------------------------------------------\n\n    def get_concept(self, concept_id: str) -> SyllabusConcept | None:\n        concept_id = str(concept_id).strip()\n        if not concept_id:\n            return None\n\n        query = text(\n            self._base_select()\n            + """\n            WHERE concept_id = :concept_id\n              AND specification_code = :specification_code\n              AND specification_version = :specification_version\n              AND is_active = TRUE\n            LIMIT 1\n            """\n        )\n\n        params = {**self._spec_params(), "concept_id": concept_id}\n        with self.engine.connect() as connection:\n            row = connection.execute(query, params).first()\n        return self._row_to_concept(row) if row else None\n\n    def get_all_concepts(self) -> list[SyllabusConcept]:\n        query = text(\n            self._base_select()\n            + """\n            WHERE specification_code = :specification_code\n              AND specification_version = :specification_version\n              AND is_active = TRUE\n            ORDER BY official_reference, concept_id\n            """\n        )\n\n        with self.engine.connect() as connection:\n            rows = connection.execute(query, self._spec_params()).all()\n        return [self._row_to_concept(row) for row in rows]\n\n    def get_concepts_by_reference(self, reference: str) -> list[SyllabusConcept]:\n        reference = str(reference).strip()\n        if not reference:\n            return []\n\n        query = text(\n            self._base_select()\n            + """\n            WHERE official_reference = :reference\n              AND specification_code = :specification_code\n              AND specification_version = :specification_version\n              AND is_active = TRUE\n            ORDER BY concept_id\n            """\n        )\n\n        params = {**self._spec_params(), "reference": reference}\n        with self.engine.connect() as connection:\n            rows = connection.execute(query, params).all()\n        return [self._row_to_concept(row) for row in rows]\n\n    def get_concepts_by_paper(self, paper: Paper) -> list[SyllabusConcept]:\n        if paper not in ("Paper 1", "Paper 2"):\n            raise ValueError("paper must be \'Paper 1\' or \'Paper 2\'")\n\n        query = text(\n            self._base_select()\n            + """\n            WHERE paper = :paper\n              AND specification_code = :specification_code\n              AND specification_version = :specification_version\n              AND is_active = TRUE\n            ORDER BY official_reference, concept_id\n            """\n        )\n\n        params = {**self._spec_params(), "paper": paper}\n        with self.engine.connect() as connection:\n            rows = connection.execute(query, params).all()\n        return [self._row_to_concept(row) for row in rows]\n\n    def get_technical_terms(self) -> tuple[str, ...]:\n        """\n        Build reusable vocabulary for preprocessing from PostgreSQL.\n\n        We include concise naming fields and aliases, but not descriptions or\n        contextual conflict terms because those are not vocabulary synonyms.\n        """\n\n        terms: set[str] = set()\n        for concept in self.get_all_concepts():\n            for value in (concept.label, concept.official_title, *concept.aliases):\n                cleaned = str(value).strip()\n                if cleaned:\n                    terms.add(cleaned)\n        return tuple(sorted(terms, key=str.casefold))\n\n    def count_concepts(self) -> int:\n        query = text(\n            """\n            SELECT COUNT(*)\n            FROM public.syllabus_concepts\n            WHERE specification_code = :specification_code\n              AND specification_version = :specification_version\n              AND is_active = TRUE\n            """\n        )\n        with self.engine.connect() as connection:\n            return int(connection.execute(query, self._spec_params()).scalar_one())\n\n    # ------------------------------------------------------------------\n    # PostgreSQL writes: authoritative storage\n    # ------------------------------------------------------------------\n\n    @staticmethod\n    def _patterns_to_json(patterns: Iterable[FlexiblePattern]) -> str:\n        return json.dumps(\n            [\n                {"label": p.label, "regex": p.regex, "weight": p.weight}\n                for p in patterns\n            ]\n        )\n\n    def upsert_concept(self, concept: SyllabusConcept) -> None:\n        """\n        Insert/update one concept in PostgreSQL.\n\n        PostgreSQL is written first. Qdrant is intentionally not changed in\n        this transaction; call sync_concept_to_qdrant() after a successful DB\n        update when the semantic representation also needs refreshing.\n        """\n\n        query = text(\n            """\n            INSERT INTO public.syllabus_concepts (\n                concept_id,\n                official_reference,\n                chapter_reference,\n                chapter_title,\n                official_title,\n                label,\n                domain,\n                description,\n                aliases,\n                paper,\n                source_pages,\n                parent_concept_id,\n                excluded_phrases,\n                ambiguous_aliases,\n                supporting_context_terms,\n                conflicting_context_terms,\n                minimum_context_hits,\n                match_patterns,\n                embedding_text,\n                specification_code,\n                specification_version,\n                is_active\n            ) VALUES (\n                :concept_id,\n                :official_reference,\n                :chapter_reference,\n                :chapter_title,\n                :official_title,\n                :label,\n                :domain,\n                :description,\n                CAST(:aliases AS JSONB),\n                :paper,\n                CAST(:source_pages AS JSONB),\n                :parent_concept_id,\n                CAST(:excluded_phrases AS JSONB),\n                CAST(:ambiguous_aliases AS JSONB),\n                CAST(:supporting_context_terms AS JSONB),\n                CAST(:conflicting_context_terms AS JSONB),\n                :minimum_context_hits,\n                CAST(:match_patterns AS JSONB),\n                :embedding_text,\n                :specification_code,\n                :specification_version,\n                :is_active\n            )\n            ON CONFLICT (concept_id) DO UPDATE SET\n                official_reference = EXCLUDED.official_reference,\n                chapter_reference = EXCLUDED.chapter_reference,\n                chapter_title = EXCLUDED.chapter_title,\n                official_title = EXCLUDED.official_title,\n                label = EXCLUDED.label,\n                domain = EXCLUDED.domain,\n                description = EXCLUDED.description,\n                aliases = EXCLUDED.aliases,\n                paper = EXCLUDED.paper,\n                source_pages = EXCLUDED.source_pages,\n                parent_concept_id = EXCLUDED.parent_concept_id,\n                excluded_phrases = EXCLUDED.excluded_phrases,\n                ambiguous_aliases = EXCLUDED.ambiguous_aliases,\n                supporting_context_terms = EXCLUDED.supporting_context_terms,\n                conflicting_context_terms = EXCLUDED.conflicting_context_terms,\n                minimum_context_hits = EXCLUDED.minimum_context_hits,\n                match_patterns = EXCLUDED.match_patterns,\n                embedding_text = EXCLUDED.embedding_text,\n                specification_code = EXCLUDED.specification_code,\n                specification_version = EXCLUDED.specification_version,\n                is_active = EXCLUDED.is_active,\n                updated_at = NOW()\n            """\n        )\n\n        params = {\n            "concept_id": concept.concept_id,\n            "official_reference": concept.official_reference,\n            "chapter_reference": concept.chapter_reference,\n            "chapter_title": concept.chapter_title,\n            "official_title": concept.official_title,\n            "label": concept.label,\n            "domain": concept.domain,\n            "description": concept.description,\n            "aliases": json.dumps(list(concept.aliases)),\n            "paper": concept.paper,\n            "source_pages": json.dumps(list(concept.source_pages)),\n            "parent_concept_id": concept.parent_concept_id,\n            "excluded_phrases": json.dumps(list(concept.excluded_phrases)),\n            "ambiguous_aliases": json.dumps(list(concept.ambiguous_aliases)),\n            "supporting_context_terms": json.dumps(\n                list(concept.supporting_context_terms)\n            ),\n            "conflicting_context_terms": json.dumps(\n                list(concept.conflicting_context_terms)\n            ),\n            "minimum_context_hits": concept.minimum_context_hits,\n            "match_patterns": self._patterns_to_json(concept.match_patterns),\n            "embedding_text": concept.embedding_text,\n            "specification_code": concept.specification_code,\n            "specification_version": concept.specification_version,\n            "is_active": concept.is_active,\n        }\n\n        with self.engine.begin() as connection:\n            connection.execute(query, params)\n\n    # ------------------------------------------------------------------\n    # Qdrant: semantic index derived from PostgreSQL\n    # ------------------------------------------------------------------\n\n    @property\n    def qdrant(self) -> QdrantClient:\n        if self._qdrant_client is None:\n            self._qdrant_client = QdrantClient(\n                url=self.config.qdrant_url,\n                api_key=self.config.qdrant_api_key,\n                timeout=self.config.qdrant_timeout_seconds,\n            )\n        return self._qdrant_client\n\n    @staticmethod\n    def point_id_for_concept(concept_id: str) -> str:\n        return str(uuid.uuid5(QDRANT_POINT_NAMESPACE, concept_id))\n\n    def qdrant_collection_exists(self) -> bool:\n        return self.qdrant.collection_exists(\n            collection_name=self.config.qdrant_collection\n        )\n\n    def ensure_qdrant_collection(self, *, recreate: bool = False) -> None:\n        exists = self.qdrant_collection_exists()\n\n        if recreate and exists:\n            self.qdrant.delete_collection(\n                collection_name=self.config.qdrant_collection\n            )\n            exists = False\n\n        if exists:\n            collection = self.qdrant.get_collection(\n                collection_name=self.config.qdrant_collection\n            )\n            vectors_config = collection.config.params.vectors\n            if not isinstance(vectors_config, models.VectorParams):\n                raise RuntimeError(\n                    "Expected one unnamed dense vector in syllabus collection."\n                )\n            expected = get_embedding_dimension(self.config.embedding_model)\n            if vectors_config.size != expected:\n                raise RuntimeError(\n                    "Qdrant vector size mismatch: "\n                    f"collection={vectors_config.size}, expected={expected}. "\n                    "Rebuild the collection after confirming the embedding model."\n                )\n            return\n\n        self.qdrant.create_collection(\n            collection_name=self.config.qdrant_collection,\n            vectors_config=models.VectorParams(\n                size=get_embedding_dimension(self.config.embedding_model),\n                distance=models.Distance.COSINE,\n            ),\n        )\n\n    def _concept_payload(self, concept: SyllabusConcept) -> dict[str, Any]:\n        """\n        Qdrant keeps enough payload for result identification/display.\n\n        PostgreSQL remains authoritative for rules and full structured data.\n        """\n\n        return {\n            "concept_id": concept.concept_id,\n            "board": "AQA",\n            "qualification": "GCSE",\n            "subject": "Computer Science",\n            "specification_code": concept.specification_code,\n            "specification_version": concept.specification_version,\n            "official_reference": concept.official_reference,\n            "chapter_reference": concept.chapter_reference,\n            "chapter_title": concept.chapter_title,\n            "official_title": concept.official_title,\n            "label": concept.label,\n            "paper": concept.paper,\n            "embedding_model": self.config.embedding_model,\n            "embedding_text": concept.embedding_text,\n        }\n\n    def sync_concept_to_qdrant(self, concept_id: str) -> None:\n        concept = self.get_concept(concept_id)\n        if concept is None:\n            raise KeyError(f"Unknown or inactive syllabus concept: {concept_id}")\n\n        self.ensure_qdrant_collection(recreate=False)\n        vector = embed_texts(\n            [concept.embedding_text], model_name=self.config.embedding_model\n        )\n        if vector.shape[0] != 1:\n            raise RuntimeError("Expected exactly one concept embedding")\n\n        self.qdrant.upsert(\n            collection_name=self.config.qdrant_collection,\n            points=[\n                models.PointStruct(\n                    id=self.point_id_for_concept(concept.concept_id),\n                    vector=vector[0].tolist(),\n                    payload=self._concept_payload(concept),\n                )\n            ],\n            wait=True,\n        )\n\n    def sync_qdrant(\n        self,\n        *,\n        recreate: bool = False,\n        batch_size: int = 32,\n    ) -> int:\n        """Rebuild/upsert Qdrant directly from PostgreSQL concepts."""\n\n        concepts = self.get_all_concepts()\n        if not concepts:\n            raise RuntimeError("PostgreSQL returned zero active syllabus concepts")\n\n        self.ensure_qdrant_collection(recreate=recreate)\n\n        embeddings = embed_texts(\n            [concept.embedding_text for concept in concepts],\n            model_name=self.config.embedding_model,\n            batch_size=batch_size,\n        )\n        if len(embeddings) != len(concepts):\n            raise RuntimeError(\n                "Number of embeddings does not match PostgreSQL concept count"\n            )\n\n        points = [\n            models.PointStruct(\n                id=self.point_id_for_concept(concept.concept_id),\n                vector=embedding.tolist(),\n                payload=self._concept_payload(concept),\n            )\n            for concept, embedding in zip(concepts, embeddings, strict=True)\n        ]\n\n        self.qdrant.upload_points(\n            collection_name=self.config.qdrant_collection,\n            points=points,\n            batch_size=batch_size,\n            parallel=1,\n            max_retries=3,\n            wait=True,\n        )\n        return len(points)\n\n    def semantic_search(\n        self,\n        text_value: str,\n        *,\n        top_k: int | None = None,\n    ) -> list[SemanticConceptMatch]:\n        cleaned = str(text_value).strip()\n        if not cleaned:\n            return []\n\n        vector = embed_texts(\n            [cleaned], model_name=self.config.embedding_model\n        )\n        if vector.shape[0] != 1:\n            return []\n        return self.search_by_vectors(vector, top_k=top_k)[0]\n\n    def search_by_vectors(\n        self,\n        query_vectors: np.ndarray,\n        *,\n        top_k: int | None = None,\n    ) -> list[list[SemanticConceptMatch]]:\n        if query_vectors.ndim != 2:\n            raise ValueError("query_vectors must be a 2D NumPy array")\n        if query_vectors.size == 0:\n            return []\n        if not self.qdrant_collection_exists():\n            raise RuntimeError(\n                "Qdrant syllabus collection does not exist. "\n                "Run sync_qdrant() first."\n            )\n\n        result_limit = top_k or self.config.qdrant_top_k\n        all_matches: list[list[SemanticConceptMatch]] = []\n\n        for vector in query_vectors:\n            response = self.qdrant.query_points(\n                collection_name=self.config.qdrant_collection,\n                query=vector.tolist(),\n                limit=result_limit,\n                score_threshold=self.config.qdrant_score_threshold,\n                with_payload=True,\n                with_vectors=False,\n            )\n\n            matches: list[SemanticConceptMatch] = []\n            for point in response.points:\n                payload = dict(point.payload or {})\n                concept_id = str(payload.get("concept_id", "")).strip()\n                if concept_id:\n                    matches.append(\n                        SemanticConceptMatch(\n                            concept_id=concept_id,\n                            score=float(point.score),\n                            payload=payload,\n                        )\n                    )\n            all_matches.append(matches)\n\n        return all_matches\n\n    def retrieve_concept_vectors(\n        self,\n        concept_ids: Iterable[str],\n    ) -> dict[str, np.ndarray]:\n        unique_ids = list(\n            dict.fromkeys(str(value).strip() for value in concept_ids if str(value).strip())\n        )\n        if not unique_ids:\n            return {}\n\n        points = self.qdrant.retrieve(\n            collection_name=self.config.qdrant_collection,\n            ids=[self.point_id_for_concept(value) for value in unique_ids],\n            with_payload=True,\n            with_vectors=True,\n        )\n\n        vectors: dict[str, np.ndarray] = {}\n        for point in points:\n            payload = dict(point.payload or {})\n            concept_id = str(payload.get("concept_id", "")).strip()\n            raw_vector = point.vector\n            if not concept_id or raw_vector is None:\n                continue\n            if isinstance(raw_vector, dict):\n                raise RuntimeError("Expected a single unnamed dense vector")\n            vectors[concept_id] = np.asarray(raw_vector, dtype=np.float32)\n        return vectors\n\n    def count_qdrant_points(self) -> int:\n        if not self.qdrant_collection_exists():\n            return 0\n        result = self.qdrant.count(\n            collection_name=self.config.qdrant_collection,\n            exact=True,\n        )\n        return int(result.count)\n\n    def _qdrant_concept_ids(self) -> set[str]:\n        if not self.qdrant_collection_exists():\n            return set()\n\n        concept_ids: set[str] = set()\n        offset = None\n        while True:\n            points, next_offset = self.qdrant.scroll(\n                collection_name=self.config.qdrant_collection,\n                limit=256,\n                offset=offset,\n                with_payload=["concept_id"],\n                with_vectors=False,\n            )\n            for point in points:\n                payload = dict(point.payload or {})\n                concept_id = str(payload.get("concept_id", "")).strip()\n                if concept_id:\n                    concept_ids.add(concept_id)\n            if next_offset is None:\n                break\n            offset = next_offset\n        return concept_ids\n\n    def verify_qdrant_sync(self) -> dict[str, Any]:\n        postgres_ids = {concept.concept_id for concept in self.get_all_concepts()}\n        qdrant_ids = self._qdrant_concept_ids()\n\n        missing = sorted(postgres_ids - qdrant_ids)\n        extra = sorted(qdrant_ids - postgres_ids)\n\n        return {\n            "status": "verified" if not missing and not extra else "mismatch",\n            "postgres_count": len(postgres_ids),\n            "qdrant_count": len(qdrant_ids),\n            "missing_in_qdrant": missing,\n            "extra_in_qdrant": extra,\n        }\n\n\n@lru_cache(maxsize=1)\ndef get_syllabus_store() -> SyllabusStore:\n    """Return one lazily-created store instance per Python process."""\n\n    return SyllabusStore()\n\n\ndef clear_syllabus_store_cache() -> None:\n    """Useful in tests after environment/configuration changes."""\n\n    get_syllabus_store.cache_clear()\n'

MODULE_PATH = PROJECT_ROOT / "app" / "services" / "syllabus_store.py"
MODULE_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

print("Created:", MODULE_PATH)
print("Module lines:", len(MODULE_SOURCE.splitlines()))
print("Static catalogue dependency present:", "cs_concept_catalog" in MODULE_SOURCE)


### Expected result

The final line should be `Static catalogue dependency present: False`. SyllabusStore itself must not depend on `CS_CONCEPTS` or `cs_concept_catalog.py`.


## Phase 3 — Import the new store

The store reuses `app.db.session`, `app.services.embedding_service`, and `qdrant-client`. No old catalogue module is imported.


In [ ]:
import importlib
import app.services.syllabus_store as syllabus_store_module

syllabus_store_module = importlib.reload(syllabus_store_module)

from app.services.syllabus_store import (
    FlexiblePattern, SemanticConceptMatch, SyllabusConcept,
    SyllabusStore, SyllabusStoreConfig,
)

store = SyllabusStore()

print("SyllabusStore imported successfully.")
print("Specification:", store.config.specification_code)
print("Version:", store.config.specification_version)
print("Qdrant collection:", store.config.qdrant_collection)


## Phase 4 — Verify PostgreSQL as the source of truth

For the current migration state, we expect exactly **92 active concepts** in `public.syllabus_concepts`.


In [ ]:
total = store.count_concepts()
print("Active PostgreSQL syllabus concepts:", total)
assert total == 92, (
    f"Expected 92 migrated concepts, but PostgreSQL returned {total}. "
    "Do not continue with catalogue removal until this is resolved."
)
print("PASS: PostgreSQL contains all 92 active syllabus concepts.")


## Phase 5 — Exact concept lookup

Old catalogue lookup becomes `store.get_concept(...)`. The returned object still contains aliases, paper, parent relation and contextual rules.


In [ ]:
binary_search = store.get_concept("aqa_3_1_3_binary_search")
assert binary_search is not None

print("Concept ID:", binary_search.concept_id)
print("Label:", binary_search.label)
print("Official reference:", binary_search.official_reference)
print("Paper:", binary_search.paper)
print("Aliases:", binary_search.aliases)
print("Supporting context:", binary_search.supporting_context_terms)


## Phase 6 — Grouped structured lookups

`get_concepts_by_reference("3.1.3")` should return the fine-grained searching concepts. Paper filtering remains a structured PostgreSQL lookup and does not need Qdrant.


In [ ]:
searching = store.get_concepts_by_reference("3.1.3")
print("Concepts under AQA 3.1.3:")
for concept in searching:
    print(" -", concept.concept_id, "->", concept.label)

print()
print("Paper 1 concept count:", len(store.get_concepts_by_paper("Paper 1")))
print("Paper 2 concept count:", len(store.get_concepts_by_paper("Paper 2")))


## Phase 7 — Verify the migrated catalogue rules

Expected non-empty rule counts:

| Rule field | Expected concepts |
|---|---:|
| `excluded_phrases` | 3 |
| `ambiguous_aliases` | 7 |
| `supporting_context_terms` | 7 |
| `conflicting_context_terms` | 1 |
| `match_patterns` | 1 |

This confirms that SyllabusStore reconstructs the full rule layer from PostgreSQL, not just topic names.


In [ ]:
concepts = store.get_all_concepts()
rule_counts = {
    "excluded_phrases": sum(bool(c.excluded_phrases) for c in concepts),
    "ambiguous_aliases": sum(bool(c.ambiguous_aliases) for c in concepts),
    "supporting_context_terms": sum(bool(c.supporting_context_terms) for c in concepts),
    "conflicting_context_terms": sum(bool(c.conflicting_context_terms) for c in concepts),
    "match_patterns": sum(bool(c.match_patterns) for c in concepts),
}
for field, count in rule_counts.items():
    print(f"{field:28s} -> {count}")

expected = {
    "excluded_phrases": 3,
    "ambiguous_aliases": 7,
    "supporting_context_terms": 7,
    "conflicting_context_terms": 1,
    "match_patterns": 1,
}
assert rule_counts == expected
print("\nPASS: contextual catalogue rules reconstruct correctly from PostgreSQL.")


### Inspect the conflicting-context case

Module 3 uses conflicting terms to prevent false matches. This confirms that the current conflicting-context record is available through SyllabusStore.


In [ ]:
conflicting_cases = [c for c in concepts if c.conflicting_context_terms]
for concept in conflicting_cases:
    print("Concept:", concept.concept_id)
    print("Label:", concept.label)
    print("Conflicting terms:", concept.conflicting_context_terms)


## Phase 8 — Technical vocabulary from PostgreSQL

`get_technical_terms()` replaces the catalogue dependency used for protected technical vocabulary. It collects labels, official titles and aliases; it does not treat descriptions or conflicting terms as synonyms.


In [ ]:
technical_terms = store.get_technical_terms()
print("Technical vocabulary terms:", len(technical_terms))
print("Sample:")
for term in technical_terms[:30]:
    print(" -", term)


## Phase 9 — Storage behaviour

The requested store handles **storage as well as lookup**. `upsert_concept()` writes PostgreSQL first and does not silently combine a DB transaction with a Qdrant network write.

Safe update flow:

```text
1. upsert_concept()
2. PostgreSQL commit succeeds
3. sync_concept_to_qdrant()
4. verify_qdrant_sync()
```

The next cell performs no write; it only confirms the methods exist.


In [ ]:
print("PostgreSQL write method available:", callable(store.upsert_concept))
print("Single-concept Qdrant sync available:", callable(store.sync_concept_to_qdrant))
print("Full Qdrant sync available:", callable(store.sync_qdrant))


## Phase 10 — Read-only Qdrant check

This cell does **not** rebuild or modify Qdrant. The existing collection may still have been created by the old catalogue-backed indexer; that is okay for inspection.


In [ ]:
try:
    qdrant_exists = store.qdrant_collection_exists()
    print("Qdrant collection exists:", qdrant_exists)
    if qdrant_exists:
        print("Qdrant point count:", store.count_qdrant_points())
        print("Current consistency check:")
        print(store.verify_qdrant_sync())
except Exception as exc:
    print("Qdrant check could not complete.")
    print("Reason:", type(exc).__name__, "-", exc)
    print("PostgreSQL tests remain valid; start Qdrant before semantic-index tests.")


## Phase 11 — Optional PostgreSQL → Qdrant sync

**Disabled by default.** When enabled, vectors are generated from `public.syllabus_concepts.embedding_text`, not `CS_CONCEPTS`.

During this migration, a clean rebuild can be done with `SYNC_QDRANT = True` and `RECREATE_QDRANT = True`. Keep recreate disabled until you intentionally want to rebuild the configured syllabus collection.


In [ ]:
SYNC_QDRANT = False
RECREATE_QDRANT = False

if SYNC_QDRANT:
    indexed = store.sync_qdrant(recreate=RECREATE_QDRANT, batch_size=32)
    print("Qdrant concepts indexed from PostgreSQL:", indexed)
    print(store.verify_qdrant_sync())
else:
    print("Skipped. Set SYNC_QDRANT = True when ready.")


## Phase 12 — Semantic search smoke test

Qdrant handles semantic nearest-neighbour retrieval. A classroom sentence about checking the middle item and discarding half should rank **Binary search** near the top. Qdrant returns a concept ID and score; authoritative rules can then be loaded from PostgreSQL.


In [ ]:
query_text = (
    "We check the middle item in the sorted list and discard half "
    "of the remaining values after each comparison."
)

try:
    if store.qdrant_collection_exists():
        matches = store.semantic_search(query_text, top_k=5)
        for rank, match in enumerate(matches, start=1):
            full_concept = store.get_concept(match.concept_id)
            label = full_concept.label if full_concept else match.payload.get("label")
            print(f"{rank}. {match.concept_id} | {label} | score={match.score:.4f}")
    else:
        print("Qdrant collection does not exist yet. Run the sync phase first.")
except Exception as exc:
    print("Semantic smoke test skipped:", type(exc).__name__, "-", exc)


## Phase 13 — Strict no-catalogue dependency check

The new store must not contain `cs_concept_catalog` or `CS_CONCEPTS`. The old catalogue can remain temporarily elsewhere while callers are migrated.


In [ ]:
module_text = MODULE_PATH.read_text(encoding="utf-8")
for forbidden in ("cs_concept_catalog", "CS_CONCEPTS"):
    assert forbidden not in module_text, f"Forbidden dependency found: {forbidden}"
print("PASS: SyllabusStore has no static catalogue dependency.")


## Phase 14 — Audit remaining Agent 1 catalogue dependencies

We **do not delete the old catalogue yet**. This read-only scan lists Python files that still reference `cs_concept_catalog` or `CS_CONCEPTS`. Those callers will be migrated one by one next.


In [ ]:
search_terms = ("cs_concept_catalog", "CS_CONCEPTS")
hits = []
for path in PROJECT_ROOT.rglob("*.py"):
    if "__pycache__" in path.parts or ".venv" in path.parts:
        continue
    try:
        text_value = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue
    found = [term for term in search_terms if term in text_value]
    if found:
        hits.append((path.relative_to(PROJECT_ROOT), found))

print("Python files still referencing the static catalogue:", len(hits))
for path, found in hits:
    print(" -", path, "->", ", ".join(found))


## Integration order after SyllabusStore passes

1. **`topic_candidate_extractor.py`** — replace `CS_CONCEPTS` with `store.get_all_concepts()` while keeping scoring/context logic unchanged.
2. **Qdrant syllabus indexing** — stop indexing from the static catalogue and use PostgreSQL through `store.sync_qdrant()`.
3. **HITL concept lookups** — use `store.get_concept()` and `store.get_concepts_by_reference()`.
4. **`technical_vocabulary.py`** — use `store.get_technical_terms()`.
5. **Frontend dropdowns** — load concepts through SyllabusStore.
6. **Module 1 and Module 3 notebooks** — remove embedded/static catalogue definitions.
7. **Repository-wide audit** — runtime references to `CS_CONCEPTS` / `cs_concept_catalog` must become zero.
8. **Only then delete `cs_concept_catalog.py`.**

### Final target

```text
Agent 1
  |
  v
SyllabusStore
  |-----------------------------|
  v                             v
PostgreSQL                  Qdrant
92 concepts + rules      semantic vectors
authoritative source      derived index
```


## Completion checklist

- [ ] `app/services/syllabus_store.py` exists.
- [ ] SyllabusStore imports without the static catalogue.
- [ ] PostgreSQL returns 92 active concepts.
- [ ] Binary search loads by `concept_id`.
- [ ] `3.1.3` returns its fine-grained concepts.
- [ ] rule counts are `3 / 7 / 7 / 1 / 1`.
- [ ] technical vocabulary is generated from PostgreSQL.
- [ ] Qdrant can be checked without modification.
- [ ] PostgreSQL → Qdrant sync is available but opt-in.
- [ ] semantic search works when Qdrant is running/indexed.
- [ ] the old catalogue has **not** been deleted prematurely.

Once these pass, the next task is to migrate Agent 1 callers onto SyllabusStore.
